# 🎬 Antigravity × Google Colab GPU × 20TB Google Drive — 1分钟+ 长视频生成工作站

本工作站专为 **60 秒以上 AI 长视频生成** 设计，解决单次生成时长限制与显存瓶颈：
1. **20TB 云盘永久模型缓存 (`HF_HOME`)**：数十 GB 的视频大模型权重直接缓存在您的 20TB Google Drive 中，**首次下载后永久秒加载**。
2. **首尾帧接力运镜 (`Last-Frame I2V Chaining`)**：自动提取上一镜头的最后一帧作为下一镜头的起始参考帧，配合分镜剧本 (`storyboard.json`) 实现一镜到底或跨场景叙事。
3. **分镜级断点续传 (`Checkpoint Resume`)**：每渲染完一个 5 秒分镜立即写入 Google Drive；若中途暂停或断开，重新运行会自动跳过已完成的镜头继续合成。
4. **FFmpeg 无损合成**：12 个 5 秒高清分镜完成后自动拼接为完整的 **60 秒+ `.mp4` 长视频**。

In [ ]:
# 1️⃣ 检查 GPU 硬件与显存状态 (G4 / T4 / L4 / A100)
!nvidia-smi
import torch
print('\n========================================')
print(f'PyTorch 版本: {torch.__version__}')
print(f'CUDA 是否可用: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU 型号: {torch.cuda.get_device_name(0)}')
    props = torch.cuda.get_device_properties(0)
    print(f'显存大小: {props.total_memory / (1024**3):.2f} GB')
print('========================================')

In [ ]:
# 2️⃣ 挂载 20TB Google Drive 并绑定 HuggingFace 永久模型缓存
import os
from google.colab import drive

drive.mount('/content/drive')

WORKSPACE_DIR = '/content/drive/MyDrive/Antigravity_Workspace'
OUTPUT_DIR = os.path.join(WORKSPACE_DIR, 'outputs')
HF_CACHE_DIR = os.path.join(WORKSPACE_DIR, 'hf_cache')
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(HF_CACHE_DIR, exist_ok=True)

os.environ['HF_HOME'] = HF_CACHE_DIR
os.chdir(WORKSPACE_DIR)

print(f'✅ 已进入云端共享工作区: {WORKSPACE_DIR}')
print(f'📦 模型永久缓存目录: {HF_CACHE_DIR}')
!df -h /content/drive
!ls -la

In [ ]:
# 3️⃣ 安装视频生成依赖 (Diffusers / Transformers / PyAV / FFmpeg)
!pip install -q -U diffusers transformers accelerate sentencepiece protobuf imageio imageio-ffmpeg av
print('✅ 视频生成依赖安装完成！')

In [ ]:
# 4️⃣ 启动 1分钟+ (60s) 长视频生成流水线
# 默认读取 storyboard.json (12 个 5s 连续镜头 = 60 秒)
# 如需先快速测试前 2 个镜头 (10 秒)，可将下方参数改为: --max-shots 2
# 如需切换为阿里通义万相引擎，可添加参数: --engine wan
!python3 long_video_pipeline.py --storyboard storyboard.json

In [ ]:
# 5️⃣ 在线预览已合成的长视频成片
import glob
from IPython.display import Video, display

mp4_files = sorted(glob.glob('outputs/*.mp4'))
if mp4_files:
    latest_video = mp4_files[-1]
    print(f'🎬 正在预览成片: {latest_video}')
    display(Video(latest_video, embed=True, width=768))
else:
    print('⚠️ 尚未找到合成后的 .mp4 成片，请先运行上方第 4 步。')